# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [1]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openapi-core 0.19.5 requires werkzeug<3.1.2, but you have werkzeug 3.1.8 which is incompatible.
semantic-kernel 1.28.0 requires pydantic!=2.10.0,!=2.10.1,!=2.10.2,!=2.10.3,<2.12,>=2.0, but you have pydantic 2.13.4 which is incompatible.
tensorflow-intel 2.17.0 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.5.0 which is incompatible.
tensorflow-intel 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

c:\Users\elocusteanu\Documents\FY26\GAIACADEMY\AI-For-Beginners\.venv312\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\elocusteanu\Documents\FY26\GAIACADEMY\AI-For-Beginners\.venv312\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [3]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

Wonderful! A week of great food and rich history sounds like a delightful way to spend your vacation. To tailor my recommendation perfectly, could you please share a bit more about your preferences?

1. Which regions or countries interest you, or are you open to any destination?
2. Are you comfortable with international flights or prefer destinations closer to home?
3. Do you have preferred climates—warm, mild, or cooler weather?
4. Any particular historical periods or cuisines you love?

Meanwhile, I’ll consider several options that balance culture, cuisine, and budget. Looking forward to making your dream vacation happen!


## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

We also introduce a tool that returns destination details so the agent grounds its recommendations in real data.

In [6]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

For a culture-loving traveler with a $2500 budget, I recommend these three destinations:

1. Kyoto, Japan - Known for its beautiful temples, traditional tea houses, and rich cultural heritage.
2. Rome, Italy - Famous for its ancient ruins, museums, and vibrant cultural scene.
3. Marrakech, Morocco - Offers a unique blend of Arab, Berber, and French cultural influences, with bustling markets and historic palaces.

These destinations offer rich cultural experiences and can generally be explored within your budget. Would you like more detailed information on any of these places?


In [7]:
# using pydantic to enforce structured output
structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="""You are a travel expert. Recommend destinations based on traveler preferences.
Use the get_destination_details tool.
Return exactly 3 recommendations that match the TravelRecommendations schema.""",
    tools=[get_destination_details],
    default_options={"response_format": TravelRecommendations},
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

travel_recommendations = response.value

print(travel_recommendations.model_dump_json(indent=2))
print("\nPersonalized note:", travel_recommendations.personalized_note)
print("First destination:", travel_recommendations.recommendations[0].destination)

{
  "recommendations": [
    {
      "destination": "Kyoto, Japan",
      "available": true,
      "best_season": "Spring (March to May)",
      "highlights": [
        "Historic temples and shrines",
        "Traditional tea ceremonies",
        "Geisha culture in Gion district"
      ],
      "estimated_budget_usd": 2300
    },
    {
      "destination": "Rome, Italy",
      "available": true,
      "best_season": "Spring (April to June) and Fall (September to October)",
      "highlights": [
        "Ancient Roman ruins like the Colosseum",
        "World-class museums and art galleries",
        "Vibrant street life and Italian cuisine"
      ],
      "estimated_budget_usd": 2400
    },
    {
      "destination": "Marrakech, Morocco",
      "available": true,
      "best_season": "Spring (March to May) and Fall (September to November)",
      "highlights": [
        "Historic medinas and souks",
        "Beautiful palaces and gardens",
        "Rich Berber and Arabic culture"
     

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [8]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

=== Destination Expert ===
Here is a ranked list of cultural and food-rich destinations for a week under $2500, based on general knowledge:

1. Marrakech, Morocco
   - Pros: Rich in culture with historic medinas, vibrant markets, and unique Moroccan cuisine; generally affordable for food and experiences.
   - Cons: Some areas might be touristy; language barriers in some places.

2. Bangkok, Thailand
   - Pros: World-famous street food and diverse local cuisine; temples and cultural sites abound; good value for money.
   - Cons: Can be crowded and hot; tourist areas sometimes pricey.

3. Rome, Italy
   - Pros: Iconic cultural landmarks and world-renowned cuisine; rich history and art.
   - Cons: Costs can be higher, but budget options exist; meals in tourist zones can be expensive.

4. Kyoto, Japan
   - Pros: Deep cultural experiences with temples and traditional tea ceremonies; renowned cuisine.
   - Cons: Japan generally is more expensive; budget management needed to stay under $2500.

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.